# L1 — Break a tiny chain (Week 2)

**What you will do.** Hash a receipt, mine three blocks, quietly change a
payment, watch the verifier fail, then re-mine. About **60–75 minutes**.

**Turn in.** A short PDF (Q1–Q5) plus three PNGs this notebook saves in
`lab_output/`: `L1-hash.png`, `L1-mine.png`, `L1-tamper.png`.

**Stuck?** Handbook → Session 3. Same words as class: hash, nonce,
`prev_hash`, verify.

Run **Cell → Run All**. If a cell errors, read the comment above it.
Do not paste someone else's notebook.



## Words we use today

Lost? Open **Start here**: https://salihtutun.github.io/acct5105/fundamentals.html

Read this once. If a later cell uses the word, come back here.

| Word | In this class it means |
|---|---|
| **Hash** | Fingerprint of exact bytes. One letter change → new print. |
| **Nonce** | How many guesses until the hash looks rare enough. |
| **Difficulty** | How rare (here: how many leading zeros). Stay at 3. |
| **prev_hash** | This page names the last page’s fingerprint. |
| **Verify** | Zeros still there AND each page points at the real last page. |



## 0. Folders for your screenshots

This cell makes `lab_output/` next to the notebook. Re-run it if you
moved files. Nothing secret goes in here.



In [ ]:
from pathlib import Path  # load libraries for this cell
import hashlib  # load libraries for this cell
from dataclasses import dataclass  # load libraries for this cell

import matplotlib.pyplot as plt  # load libraries for this cell

OUT = Path("lab_output")  # set OUT
OUT.mkdir(exist_ok=True)  # call the helper
print("screenshots will go to", OUT.resolve())  # print for the PDF or screen



## Q1 — A tamper-evident seal (4 pts)

Hash a short payment receipt. Change **one** character. Hash again.
The two fingerprints should look unrelated. That is the seal.



In [ ]:
def sha256(text: str) -> str:  # define helper sha256
    """SHA-256 hex digest of a UTF-8 string. Same input → same print."""  # docstring — what this helper does
    return hashlib.sha256(text.encode("utf-8")).hexdigest()  # hand back the result


receipt = "Ava pays Ben 2 EAGE for 10 min algebra"  # payment text to seal
tweaked = "Ava pays Ben 3 EAGE for 10 min algebra"  # one digit changed

h0 = sha256(receipt)  # payment text to seal
h1 = sha256(tweaked)  # one-character tamper
print("original:", h0)  # print for the PDF or screen
print("tweaked: ", h1)  # print for the PDF or screen
print("same?", h0 == h1)  # print for the PDF or screen

fig, ax = plt.subplots(figsize=(8, 3.2))  # set fig
ax.axis("off")  # call the helper
ax.set_title("L1-hash: one character, new fingerprint")  # call the helper
ax.text(0.02, 0.62, "original\n" + h0, family="monospace", fontsize=9)  # call the helper
ax.text(0.02, 0.18, "one digit changed\n" + h1, family="monospace", fontsize=9)  # call the helper
fig.savefig(OUT / "L1-hash.png", dpi=140, bbox_inches="tight")  # save lab PNG
plt.show()  # draw the chart
print("saved", OUT / "L1-hash.png")  # print for the PDF or screen



**Write in the PDF (Q1).** In finance language, what did you just show?
2–4 sentences. Hint: a stored hash is a seal on the *exact* bytes.



## Q2 — Mine three pages (4 pts)

A **nonce** is the guess counter. **Difficulty 3** means the hash must
start with `000`. Stay at 3 — this is a classroom toy, not Bitcoin.



In [ ]:
@dataclass  # decorator on the next class or function
class Block:  # define data type Block
    prev_hash: str  # fingerprint of the page before this one
    txs: str  # payment text stored on this page
    nonce: int = 0  # guess counter; mining changes this

    def header(self) -> str:  # define helper header
        return f"{self.prev_hash}|{self.txs}|{self.nonce}"  # hand back the result

    def hash(self) -> str:  # define helper hash
        return sha256(self.header())  # hand back the result


def mine(prev_hash: str, txs: str, difficulty: int = 3) -> Block:  # define helper mine
    """Brute-force nonce until the hash starts with `difficulty` zeros."""  # docstring — what this helper does
    prefix = "0" * difficulty  # leading zeros required
    nonce = 0  # mining guess counter
    while True:  # repeat until done
        cand = Block(prev_hash, txs, nonce)  # mining guess counter
        if cand.hash().startswith(prefix):  # branch when this is true
            return cand  # hand back the result
        nonce += 1  # bump the counter


DIFFICULTY = 3  # leading zeros required
genesis = mine("0" * 64, "class opens the lab", DIFFICULTY)  # leading zeros required
b2 = mine(genesis.hash(), "Ava->Ben:2", DIFFICULTY)  # leading zeros required
b3 = mine(b2.hash(), "Ben->Cara:1", DIFFICULTY)  # leading zeros required
chain = [genesis, b2, b3]  # set chain

print(f"{'i':>2}  {'nonce':>8}  hash")  # print for the PDF or screen
for i, b in enumerate(chain):  # loop each item
    print(f"{i:2d}  {b.nonce:8d}  {b.hash()}")  # print for the PDF or screen

fig, ax = plt.subplots(figsize=(8, 2.8))  # set fig
ax.axis("off")  # call the helper
ax.set_title("L1-mine: three blocks, difficulty 3")  # call the helper
lines = [f"{i}  nonce={b.nonce}  {b.hash()[:18]}…" for i, b in enumerate(chain)]  # mining guess counter
ax.text(0.02, 0.5, "\n".join(lines), family="monospace", fontsize=10, va="center")  # call the helper
fig.savefig(OUT / "L1-mine.png", dpi=140, bbox_inches="tight")  # save lab PNG
plt.show()  # draw the chart
print("saved", OUT / "L1-mine.png")  # print for the PDF or screen



**Write in the PDF (Q2).** What is a nonce, what is difficulty, and when
does a block pass? At most 8 sentences. Use what the cell printed.



## Q3 — Quiet edit, no re-mine (4 pts)

Change Ava’s payment in **block 1**. Do **not** re-mine. The verifier
should fail: either the puzzle zeros are gone, or the next page still
points at the old fingerprint.



In [ ]:
def verify_chain(chain, difficulty: int) -> bool:  # define helper verify_chain
    """PoW zeros on every block, and each parent pointer is honest."""  # docstring — what this helper does
    prefix = "0" * difficulty  # leading zeros required
    for i, block in enumerate(chain):  # loop each item
        if not block.hash().startswith(prefix):  # branch when this is true
            print("FAIL: block", i, "hash no longer meets difficulty")  # print for the PDF or screen
            return False  # hand back the result
        if i == 0:  # branch when this is true
            continue  # run this step
        if block.prev_hash != chain[i - 1].hash():  # branch when this is true
            print("FAIL: block", i, "points at a ghost parent")  # print for the PDF or screen
            return False  # hand back the result
    print("chain looks clean")  # print for the PDF or screen
    return True  # hand back the result


print("before vandalism:", verify_chain(chain, DIFFICULTY))  # print for the PDF or screen
chain[1].txs = "Ava->Ben:200"  # quiet edit — no new nonce
after = verify_chain(chain, DIFFICULTY)  # leading zeros required
print("after a quiet edit:", after)  # print for the PDF or screen

fig, ax = plt.subplots(figsize=(8, 2.4))  # set fig
ax.axis("off")  # call the helper
ax.set_title("L1-tamper: edit block 1, do not re-mine")  # call the helper
ax.text(  # run this step
    0.02,  # run this step
    0.5,  # run this step
    f"before: True\nafter:  {after}\n(expected False — PoW and/or parent link)",  # run this step
    family="monospace",  # set family
    fontsize=11,  # set fontsize
    va="center",  # set va
)  # close the bracket above
fig.savefig(OUT / "L1-tamper.png", dpi=140, bbox_inches="tight")  # save lab PNG
plt.show()  # draw the chart
print("saved", OUT / "L1-tamper.png")  # print for the PDF or screen



**Write in the PDF (Q3).** What fails first, and why? Paste the True/False
line plus two sentences.



## Q4 — Re-mine from the break (4 pts)

You must re-mine the tampered block **and every block after it**. On a
long public chain that cost is the security budget.



In [ ]:
# Rebuild from block 1 forward. Genesis stays. Tampered txs stay.
repaired = [chain[0]]  # set repaired
n_remined = 0  # set n remined
for old in chain[1:]:  # loop each item
    nxt = mine(repaired[-1].hash(), old.txs, DIFFICULTY)  # leading zeros required
    repaired.append(nxt)  # call the helper
    n_remined += 1  # bump the counter

print("re-mined", n_remined, "blocks (tampered + everything after)")  # print for the PDF or screen
print("verifies again?", verify_chain(repaired, DIFFICULTY))  # print for the PDF or screen
print("new hashes:")  # print for the PDF or screen
for i, b in enumerate(repaired):  # loop each item
    print(i, b.hash()[:20], "txs=", b.txs)  # print for the PDF or screen



**Write in the PDF (Q4).** How many blocks did you re-mine? Why is that
expensive on a long public chain? Four sentences. Not a sermon about electricity.



## Q5 — Unbundle for *your* coin (4 pts)

Edit the two lists. Class ERC-20s sit on someone else’s chain.



In [ ]:
# Fill both lists in your own words, e.g. used_here = ["...", "..."].
# Left: pieces this notebook actually used. Right: Bitcoin pieces a class ERC-20 does not need.
used_here = []  # set used here
not_needed_to_issue_an_erc20 = []  # set not needed to issue an erc20
print("We used:", ", ".join(used_here) or "(empty: fill this list)")  # print for the PDF or screen
print("We did not need, to issue a class ERC-20:", ", ".join(not_needed_to_issue_an_erc20) or "(empty: fill this list)")  # print for the PDF or screen
print("PDF: pick TWO from each list and say why in a sentence each.")  # print for the PDF or screen



## Checklist before you export the PDF

1. `lab_output/L1-hash.png`, `L1-mine.png`, `L1-tamper.png` exist.
2. Q1–Q5 are in **your** words (talking with a teammate is OK).
3. Optional extra: Naranjo *Proof of Work Example* if you want another
   implementation of the same verbs —
   https://www.lorenzonaranjo.com/class-materials/fintech/notebooks.html

